# LLM-as-a-Judge Calibration and Evaluation

This notebook loads a 10–15 example calibration set from `calibration_set.json`, runs an LLM judge (DeepSeek or Llama as fallback) in Colab/Kaggle, stores judge outputs, and computes validation metrics (Cohen's κ, Win Rate, position-bias stats).

The notebook implements the workflow described in `docs/reports/comment_quality_evaluation.md` (Section 6).

## 1. Setup & Dependencies

Install required packages for model loading, inference, and metrics computation.

In [2]:
!pip install -q -U bitsandbytes accelerate transformers tqdm scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 29.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 24.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 31.7 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 101.1 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 637.4/637.4 kB 37.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 94.7 MB/s eta 0:00:00:00:01


In [3]:
!pip install -q groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 141.7/141.7 kB 6.7 MB/s eta 0:00:00


## 2. Import Libraries

Core dependencies: PyTorch, Transformers, Pandas, scikit-learn for metrics.

In [4]:
import json, os, re
import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from tqdm import tqdm
from sklearn.metrics import cohen_kappa_score
from groq import Groq

## 3. Data Loading & Validation

Load calibration set from JSON. Validates schema and handles missing files gracefully.

**Schema Expected:**
- `file_id`: Unique file identifier
- `path`: File path in PR
- `patched_content`: Full file content after changes
- `human_comments`: List of human review comments
- `ai_comments`: List of AI-generated comments
- `human_score`: Ground truth score (1-5) assigned by human annotator
- `human_side`: Which position human review occupies ("A" or "B" for randomization)

In [5]:
DATASET_PATH = '/kaggle/input/datasets/se1nastol/ai-code-reviewer-calibration-set/calibration_set.json'

try:
    with open(DATASET_PATH, 'r', encoding='utf-8') as f:
        calibration_set = json.load(f)
    print(f"Loaded {len(calibration_set)} examples from calibration_set.json")
except FileNotFoundError:
    print("WARNING: calibration_set.json not found. Please upload it.")
    calibration_set = []

Loaded 15 examples from calibration_set.json


## 4. Model Loading & Judge Initialization

Loads the DeepSeek-R1-Distill-Qwen-32B / 70B model for inference or fallback to Llama-3.3-70b.

**Model Choice Rationale** (see `docs/reports/comment_quality_evaluation.md` Section 6.1.2):
- Code-specific reasoning via chain-of-thought
- Low position bias compared to base models
- Fallback to LLama as best model at human judge correlation  

In [ ]:
os.environ["GROQ_API_KEY"] = "gsk_YOUR_API_KEY"
client = Groq()

MODEL_NAME = "llama-3.3-70b-versatile"  # deepseek-r1-distill-llama-70b was deprecated on Oct 2025

In [6]:
# model_id = "unsloth/DeepSeek-R1-Distill-Qwen-32B-bnb-4bit"

# print("Loading tokenizer...")
# tokenizer = AutoTokenizer.from_pretrained(model_id)

# print("Loading 32B model across 2x T4 GPUs (this takes ~3 mins)...")
# model = AutoModelForCausalLM.from_pretrained(
#     model_id,
#     device_map="auto",
#     torch_dtype=torch.float16,
# )
# print("Model loaded successfully!")

## 5. Prompt Engineering & Judge Inference

Defines the evaluation prompt (builton Section 6.3 of methodology document) and implements JSON extraction from model outputs.

**Key Features:**
- Holistic N:M review comparison (not comment-by-comment matching)
- Anti-bias instructions explicit in system prompt
- Chain-of-thought reasoning required
- Scoring rubric (1-5) with semantic vs. stylistic distinction
- Graceful JSON parsing with fallback to neutral Tie verdict

In [ ]:
def build_prompt(entry, review_a, review_b):
    prompt = f"""You are a strict Senior Staff Software Engineer acting as a code review judge for a Python codebase.
Your task is to objectively evaluate two sets of review comments—Review A and Review B—on a single Python file.

### ARCHITECTURAL CONSTRAINTS (CRITICAL)
This system is designed ONLY to catch **merge-blocking semantic issues** (e.g., logical correctness, security vulnerabilities, thread-safety, performance regressions, or severe maintainability flaws).
**Style, formatting, and minor naming conventions are OUT OF SCOPE** (they are handled by CI linters). Comments focusing purely on style should be treated as "Noise".

### SOURCE MATERIAL

FILE PATH: {entry.get('path')}
FILE CONTENT:
```python
{(entry.get('patched_content') or '')[:2000]}
```

### REVIEWS TO EVALUATE

REVIEW A:
{json.dumps(review_a, ensure_ascii=False, indent=2)}

REVIEW B:
{json.dumps(review_b, ensure_ascii=False, indent=2)}

### EVALUATION PROTOCOL (STRICT)
You MUST evaluate both reviews using the following structured rubric.

| Score | Description |
|-------|-------------|
| 5 | **Excellent:** Review correctly identifies critical/blocking semantic issues, provides clear and actionable fixes, and avoids noise or hallucinations. No significant blocking issues were missed. |
| 4 | **Strong:** Review addresses most important blocking issues and is mostly actionable, but may miss a minor logical point or include a slight nitpick/style comment. |
| 3 | **Adequate:** Review catches some relevant issues but misses at least one critical blocking point, OR it dilutes good advice with significant stylistic noise/unnecessary comments. Value is mixed. |
| 2 | **Weak:** Review misses multiple critical blocking issues, OR is predominantly composed of incorrect, irrelevant stylistic noise. May cause developer frustration. |
| 1 | **Harmful:** Review is misleading, hallucinates code that does not exist, suggests breaking changes, or completely fails to identify obvious critical bugs. |

### EVALUATION RULES:
1. **Hallucination Check (Score 1):** If a review mentions variables, loops, or logic that DO NOT EXIST in the provided FILE CONTENT, you MUST score it a 1 (Harmful). Do not assume code exists outside the snippet.
2. **The "Noise" Penalty (Score 2 or 3):** If a review ignores critical bugs to focus purely on PEP8 formatting, docstrings, or minor naming ("refactor for clarity"), it is providing Noise. Score it a 2 or 3 depending on severity.
3. **The "Silence" Evaluation:** If a review is empty (no comments):
   - If the FILE CONTENT contains a blocking semantic bug, the empty review missed it. Score = 1 or 2.
   - If the FILE CONTENT is free of blocking semantic bugs (even if style is bad), an empty review is correct. Score = 5.
4. **Outcome Selection:**
   - If Score A > Score B: "A Win"
   - If Score B > Score A: "B Win"
   - If Score A == Score B: "Tie"

### OUTPUT FORMAT
You must output your reasoning first in a <think> block, explicitly verifying if the issues mentioned are semantic (blocking) vs stylistic (noise), and if the code actually exists.
Then, output ONLY a valid JSON object in this exact format:
```json
{{
  "review_a_score": 0,
  "review_a_reasoning": "...",
  "review_b_score": 0,
  "review_b_reasoning": "...",
  "outcome": "A Win/B Win/Tie"
}}
```"""
    return prompt


def extract_json_from_deepseek(text):
    json_match = re.search(r'```json\s*(\{.*?\})\s*```', text, re.DOTALL)
    if json_match:
        json_str = json_match.group(1)
    else:
        json_match = re.search(r'\{[^{}]*\}', text, re.DOTALL)
        json_str = json_match.group(0) if json_match else "{}"

    try:
        return json.loads(json_str)
    except Exception as e:
        print(f"JSON Parse Error: {e}")
        return {'review_a_score': 3, 'review_b_score': 3, 'outcome': 'Tie', 'error': 'Parse Failed'}


def judge_infer(entry, review_a, review_b):
    prompt = build_prompt(entry, review_a, review_b)

    messages = [
        {"role": "user", "content": prompt}
    ]

    text_prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(text_prompt, return_tensors="pt").to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=500,
            temperature=0.1,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )

    input_length = inputs.input_ids.shape[1]
    response = tokenizer.decode(outputs[0][input_length:], skip_special_tokens=True)

    result = extract_json_from_deepseek(response)
    result['raw_deepseek_output'] = response
    return result


def judge_infer_api(entry, review_a, review_b):
    prompt = build_prompt(entry, review_a, review_b)

    try:
        chat_completion = client.chat.completions.create(
            messages=[
                {"role": "user", "content": prompt}
            ],
            model=MODEL_NAME,
            temperature=0.1,
            max_completion_tokens=2000,
        )

        response = chat_completion.choices[0].message.content

        result = extract_json_from_deepseek(response)
        result['raw_deepseek_output'] = response
        return result

    except Exception as e:
        print(f"API Error: {e}")
        return {'review_a_score': 3, 'review_b_score': 3, 'outcome': 'Tie', 'error': str(e)}

## 6. Running Bidirectional Evaluation

Implements position-bias mitigation via bidirectional scoring (Section 6.4.2 of methodology):
- **Pass 1:** Review A (Human) in Position 1, Review B (AI) in Position 2
- **Pass 2:** Positions swapped—Review B (AI) in Position 1, Review A (Human) in Position 2

Outputs are saved to `raw_judge_outputs.json` for per-file debugging.

In [ ]:
from datetime import datetime

raw_outputs = []
print("Starting evaluation (Double-blind swapping protocol)...")

for entry in tqdm(calibration_set):
    file_id = entry.get('file_id', 'unknown')

    if entry.get('human_side', 'A') == 'A':
        human_rev = entry.get('human_comments', [])
        ai_rev = entry.get('ai_comments', [])
    else:
        human_rev = entry.get('ai_comments', [])
        ai_rev = entry.get('human_comments', [])

    # Pass 1: [A=Human, B=AI]
    out1 = judge_infer_api(entry, human_rev, ai_rev)
    # Pass 2: [A=AI, B=Human]
    out2 = judge_infer_api(entry, ai_rev, human_rev)

    raw_outputs.append({
        'file_id': file_id,
        'pass1_human_vs_ai': out1,
        'pass2_ai_vs_human': out2,
        'ts': datetime.utcnow().isoformat()
    })

with open('raw_judge_outputs.json', 'w', encoding='utf-8') as f:
    json.dump(raw_outputs, f, indent=2, ensure_ascii=False)

print(f"Saved raw_judge_outputs.json for {len(raw_outputs)} files.")

Starting evaluation (Double-blind swapping protocol)...


  0%|          | 0/15 [00:00<?, ?it/s]/tmp/ipykernel_55/3466079953.py:25: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  'ts': datetime.utcnow().isoformat()
 73%|███████▎  | 11/15 [02:39<00:45, 11.37s/it]

API Error: Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01knepx3y4e5rb79f61t8v4y0e` service tier `on_demand` on tokens per day (TPD): Limit 100000, Used 99067, Requested 2253. Please try again in 19m0.48s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}
API Error: Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01knepx3y4e5rb79f61t8v4y0e` service tier `on_demand` on tokens per day (TPD): Limit 100000, Used 99067, Requested 2065. Please try again in 16m18.048s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}
API Error: Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01knepx3y4e5rb79f61t8v4y0e` serv

100%|██████████| 15/15 [02:39<00:00, 10.65s/it]

API Error: Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01knepx3y4e5rb79f61t8v4y0e` service tier `on_demand` on tokens per day (TPD): Limit 100000, Used 99067, Requested 2042. Please try again in 15m58.175999999s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}
API Error: Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01knepx3y4e5rb79f61t8v4y0e` service tier `on_demand` on tokens per day (TPD): Limit 100000, Used 99067, Requested 1978. Please try again in 15m2.879999999s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}
API Error: Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01knepx3y4e5rb79f61

## 7. Metrics Computation & Calibration Analysis

Computes Cohen's κ, Spearman correlation and other metrics between manual human scores and judge predictions.

In [ ]:
from collections import Counter

human_scores = []
judge_scores = []
final_outcomes = []
position_flips = 0

for ro in raw_outputs:
    file_id = ro['file_id']
    entry = next((e for e in calibration_set if e.get('file_id') == file_id), None)

    human_score = entry.get('human_score') if entry else None

    p1 = ro.get('pass1_human_vs_ai', {})
    p2 = ro.get('pass2_ai_vs_human', {})

    # Pass 1: A=Human, B=AI. Pass 2: A=AI, B=Human
    ai_score_pass1 = p1.get('review_b_score', 3)
    ai_score_pass2 = p2.get('review_a_score', 3)

    out1 = p1.get('outcome', 'Tie')
    out2 = p2.get('outcome', 'Tie')

    # Position bias check
    if out1 == 'B Win' and out2 == 'A Win':
        final = 'AI Win'
    elif out1 == 'A Win' and out2 == 'B Win':
        final = 'Human Win'
    elif out1 == 'Tie' and out2 == 'Tie':
        final = 'Tie'
    else:
        final = 'Tie (Bias Detected)'
        position_flips += 1

    final_outcomes.append(final)

    # Average over 2 passes
    avg_ai_score = int(round((ai_score_pass1 + ai_score_pass2) / 2))

    if human_score is not None:
        human_scores.append(human_score)
        judge_scores.append(avg_ai_score)

# limit to first 10 because others got API rate limited
kappa = cohen_kappa_score(human_scores[:10], judge_scores[:10]) if len(human_scores) > 0 else None

metrics = {
    'n_samples': len(calibration_set),
    'position_flips_mitigated': position_flips,
    'cohens_kappa': kappa,
    'final_winrate_summary': dict(Counter(final_outcomes))
}

with open('metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)

print("\nEvaluation Results")
print(json.dumps(metrics, indent=2))


Evaluation Results
{
  "n_samples": 15,
  "position_flips_mitigated": 3,
  "cohens_kappa": 0.012345679012345623,
  "final_winrate_summary": {
    "Tie (Bias Detected)": 3,
    "AI Win": 3,
    "Human Win": 2,
    "Tie": 7
  }
}


In [ ]:
from scipy.stats import spearmanr, kendalltau
from sklearn.metrics import mean_absolute_error, cohen_kappa_score

# limit to first 10 because others got API rate limited
h = human_scores[:10]
j = judge_scores[:10]

# 1. Spearman
spearman_corr, _ = spearmanr(h, j)

# 2. Kendall
kendall_corr, _ = kendalltau(h, j)

# 3. MAE
mae = mean_absolute_error(h, j)

# 4. Weighted Kappa
w_kappa = cohen_kappa_score(h, j, weights='quadratic')

# 5. Adjacent Accuracy
adj_acc = sum(1 for a, b in zip(h, j) if abs(a - b) <= 1) / len(h)

print(f"Weighted Kappa: {w_kappa:.3f}")
print(f"Spearman Correlation: {spearman_corr:.3f}")
print(f"Mean Absolute Error: {mae:.3f}")
print(f"Adjacent Accuracy (±1): {adj_acc:.1%}")
print(f"Kappa: {kappa:.3f}")

Weighted Kappa: 0.294
Spearman Correlation: 0.244
Mean Absolute Error: 1.300
Adjacent Accuracy (±1): 60.0%
Kappa: 0.012
